# 02 — Cleaning & Feature Engineering

This notebook executes the decisions surfaced in `01_eda.ipynb` and produces a modeling-ready dataset.

## Pipeline

1. **Setup & load** — import functions from `src/data.py`
2. **Cohort definition** — exclude expired/hospice encounters; deduplicate by patient (keep first encounter)
3. **Drop unusable columns** — `weight`, zero-variance medications, identifiers
4. **Treat missing as category** — for `A1Cresult`, `max_glu_serum`, `medical_specialty`, `payer_code`, and disguised missingness in `admission_type_id`
5. **Engineer clinical features** — A1C/glucose measurement flags, prior healthcare utilization, medication change counts, ICD-9 grouped into clinical chapters
6. **Encode categoricals** — `age` as ordered ordinal, one-hot for the rest, target binarized
7. **Sanity checks & save** — write `X.parquet` and `y.parquet` to `data/processed/`

## Cohort rationale

Patients with discharge dispositions `Expired` or any `Hospice` category are excluded — they cannot be modeled for readmission prevention. Among the remaining encounters, we keep only the **first encounter per patient** to prevent leakage when the same patient appears in both train and test splits. This mirrors the methodology of Strack et al. (2014).

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Make src/ importable (it lives at the project root, one level up from notebooks/)
sys.path.append("..")
from src.data import load_diabetic_data, load_ids_mapping

# Display options for the cleaning work
pd.set_option("display.max_columns", 100)

# Load raw data
df = load_diabetic_data()
id_maps = load_ids_mapping()

print(f"Raw dataset: {df.shape[0]:,} encounters × {df.shape[1]} columns")
print(f"Unique patients: {df['patient_nbr'].nunique():,}")

Raw dataset: 101,766 encounters × 50 columns
Unique patients: 71,518


/Users/rsantos43/Documents/analise-de-dados/projetos/diabetes-readmission/notebooks/../src/data.py:22: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, na_values="?")


In [2]:
# Step 1 of cohort definition: exclude expired/hospice encounters
EXPIRED_OR_HOSPICE = [11, 13, 14, 19, 20, 21]

before = len(df)
df = df[~df["discharge_disposition_id"].isin(EXPIRED_OR_HOSPICE)].copy()
after = len(df)

print(f"Excluded {before - after:,} expired/hospice encounters")
print(f"After exclusion: {after:,} encounters")

Excluded 2,423 expired/hospice encounters
After exclusion: 99,343 encounters


In [3]:
# Step 2 of cohort definition: keep first encounter per patient
# Sorting by encounter_id (ascending) keeps the chronologically first
# encounter, since encounter_id increases monotonically with time.

before = len(df)
df = (
    df.sort_values("encounter_id")
      .drop_duplicates(subset="patient_nbr", keep="first")
      .copy()
)
after = len(df)

print(f"Removed {before - after:,} repeat encounters")
print(f"Modeling cohort: {after:,} unique patients")

Removed 29,353 repeat encounters
Modeling cohort: 69,990 unique patients


### Final modeling cohort

- Excluded 2,423 encounters with discharge disposition Expired or Hospice
- Deduplicated to first encounter per patient (29,353 repeat encounters removed)
- **Final cohort: 69,990 unique patients** (~69% of the original dataset)

The drop from 71,518 unique patients in the raw data to 69,990 here reflects patients whose only encounter ended in death or hospice care — they were removed in the previous step. Each remaining row represents one independent patient, which eliminates leakage at the deduplication stage and allows a standard random split downstream without `GroupKFold`.

In [4]:
# Drop columns with no modeling value:
# - weight: 97% missing
# - examide, citoglipton: zero variance (single value across all rows)
# - encounter_id: identifier
# - patient_nbr: identifier (we already deduplicated by it)

cols_to_drop = ["weight", "examide", "citoglipton", "encounter_id", "patient_nbr"]
df = df.drop(columns=cols_to_drop)

print(f"Dropped: {cols_to_drop}")
print(f"Remaining: {df.shape[1]} columns")

Dropped: ['weight', 'examide', 'citoglipton', 'encounter_id', 'patient_nbr']
Remaining: 45 columns


In [5]:
# Variables where missingness is informative (MNAR) or where we don't want
# to impute artificially. Encode missing as a literal category.

missing_as_category = {
    "A1Cresult": "Not measured",
    "max_glu_serum": "Not measured",
    "medical_specialty": "Unknown",
    "payer_code": "Unknown",
    "race": "Unknown",  # ~2% missing — small enough that "Unknown" is fine
}

for col, label in missing_as_category.items():
    df[col] = df[col].fillna(label)

# Check that there's no residual missing in these columns
print("Residual missing in handled columns:")
print(df[list(missing_as_category)].isna().sum())

Residual missing in handled columns:
A1Cresult            0
max_glu_serum        0
medical_specialty    0
payer_code           0
race                 0
dtype: int64


In [6]:
# admission_type_id has disguised missingness: NULL, Not Available, Not Mapped
# all represent missing data. Consolidate into a single "Unknown" category.

# First, map IDs to descriptions
df["admission_type"] = df["admission_type_id"].map(id_maps["admission_type_id"])

# Then collapse the three "missing-like" categories into one
disguised_missing = ["NULL", "Not Available", "Not Mapped"]
df["admission_type"] = df["admission_type"].replace(
    {v: "Unknown" for v in disguised_missing}
)

# Drop the original ID column — we'll use the description column going forward
df = df.drop(columns=["admission_type_id"])

print(df["admission_type"].value_counts())

admission_type
Emergency        35480
Elective         13787
Urgent           12803
Unknown           7893
Trauma Center       18
Newborn              9
Name: count, dtype: int64


### Cleaning summary

- **Dropped** 5 columns: `weight` (97% missing), `examide` and `citoglipton` (zero variance), `encounter_id` and `patient_nbr` (identifiers no longer needed after deduplication).
- **Missing as category** applied to 5 columns: `A1Cresult` and `max_glu_serum` ("Not measured"), `medical_specialty` and `payer_code` ("Unknown"), `race` ("Unknown").
- **Consolidated disguised missingness** in `admission_type_id`: `NULL`, `Not Available`, and `Not Mapped` collapsed into "Unknown".
- The original `admission_type_id` was replaced with its readable counterpart `admission_type`.

Next: engineer clinical features.

In [7]:
# Feature 1: a1c_measured — binary flag for whether A1C was ordered.
# Per the EDA, the act of measuring A1C is a stronger signal than the value.
df["a1c_measured"] = (df["A1Cresult"] != "Not measured").astype(int)

# Feature 2: glu_measured — same logic for max_glu_serum.
df["glu_measured"] = (df["max_glu_serum"] != "Not measured").astype(int)

# Feature 3: total_visits_prior — sum of healthcare utilization in the year
# before the index admission. A composite proxy for chronic disease burden.
df["total_visits_prior"] = (
    df["number_outpatient"]
    + df["number_emergency"]
    + df["number_inpatient"]
)

# Quick sanity checks
print("a1c_measured distribution:")
print(df["a1c_measured"].value_counts())
print("\nglu_measured distribution:")
print(df["glu_measured"].value_counts())
print("\ntotal_visits_prior — stats:")
print(df["total_visits_prior"].describe().round(2))

a1c_measured distribution:
a1c_measured
0    57144
1    12846
Name: count, dtype: int64

glu_measured distribution:
glu_measured
0    66641
1     3349
Name: count, dtype: int64

total_visits_prior — stats:
count    69990.00
mean         0.56
std          1.43
min          0.00
25%          0.00
50%          0.00
75%          1.00
max         49.00
Name: total_visits_prior, dtype: float64


In [8]:
# n_meds_changed: count of antidiabetic medications with status "Up" or "Down"
# during the encounter. Reflects active diabetes management.

# All 23 antidiabetic medication columns (after dropping examide and citoglipton)
med_cols = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide",
    "glimepiride", "acetohexamide", "glipizide", "glyburide",
    "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose",
    "miglitol", "troglitazone", "tolazamide", "insulin",
    "glyburide-metformin", "glipizide-metformin",
    "glimepiride-pioglitazone", "metformin-rosiglitazone",
    "metformin-pioglitazone",
]

# Each med column has values: "No", "Steady", "Up", "Down"
# Count, per row, how many meds were "Up" or "Down"
df["n_meds_changed"] = df[med_cols].isin(["Up", "Down"]).sum(axis=1)

print("n_meds_changed distribution:")
print(df["n_meds_changed"].value_counts().sort_index())
print(f"\nMean: {df['n_meds_changed'].mean():.2f}")
print(f"Patients with at least one med change: "
      f"{(df['n_meds_changed'] > 0).sum():,} "
      f"({(df['n_meds_changed'] > 0).mean()*100:.1f}%)")

n_meds_changed distribution:
n_meds_changed
0    52739
1    16255
2      919
3       74
4        3
Name: count, dtype: int64

Mean: 0.26
Patients with at least one med change: 17,251 (24.6%)


### Engineered features (so far)

- **`a1c_measured`** (binary): 1 if A1C was ordered during the encounter, regardless of result. Per EDA, this is hypothesized to be a stronger predictor than the A1C value itself, capturing care quality.
- **`glu_measured`** (binary): 1 if serum glucose was measured. Same logic.
- **`total_visits_prior`**: sum of outpatient + emergency + inpatient visits in the year before the index admission. Composite proxy for chronic disease burden and healthcare-seeking behavior.
- **`n_meds_changed`**: count of antidiabetic medications with status "Up" or "Down" during the encounter. Reflects whether the inpatient team actively adjusted the diabetes regimen — another signal of active diabetes management.

In [9]:
"""Feature engineering: ICD-9 chapter grouping, target binarization,
and other clinical transformations.
"""
from __future__ import annotations

import pandas as pd


def binarize_readmission(series: pd.Series) -> pd.Series:
    """Convert the 3-class target into binary.

    1 = readmission within 30 days (clinical outcome of interest)
    0 = readmission after 30 days OR no readmission
    """
    return (series == "<30").astype(int)


# ICD-9-CM chapter boundaries (numeric prefix → chapter name).
# Source: official ICD-9-CM tabular list.
# Special codes (V-codes for health-status factors, E-codes for external causes)
# get their own categories.
ICD9_CHAPTERS = [
    (1, 139, "Infectious"),
    (140, 239, "Neoplasms"),
    (240, 279, "Endocrine"),         # includes 250.x diabetes
    (280, 289, "Blood"),
    (290, 319, "Mental"),
    (320, 389, "Nervous"),
    (390, 459, "Circulatory"),
    (460, 519, "Respiratory"),
    (520, 579, "Digestive"),
    (580, 629, "Genitourinary"),
    (630, 679, "Pregnancy"),
    (680, 709, "Skin"),
    (710, 739, "Musculoskeletal"),
    (740, 759, "Congenital"),
    (760, 779, "Perinatal"),
    (780, 799, "Symptoms"),          # ill-defined conditions
    (800, 999, "Injury"),
]


def group_icd9(code: object) -> str:
    """Map a single ICD-9 code (as stored in this dataset) to a clinical chapter.

    The dataset stores codes as strings like "250.83", "428.0", "V58", "E885".
    V-codes (factors influencing health status) and E-codes (external causes)
    are grouped into their own categories.
    """
    if pd.isna(code):
        return "Missing"

    code_str = str(code).strip()

    # V-codes: factors influencing health status (e.g., V58 = aftercare)
    if code_str.startswith("V"):
        return "V_code"

    # E-codes: external causes of injury (e.g., E885 = fall)
    if code_str.startswith("E"):
        return "E_code"

    # Numeric codes: take the integer part before the decimal
    try:
        numeric = int(float(code_str))
    except (ValueError, TypeError):
        return "Unknown"

    for low, high, chapter in ICD9_CHAPTERS:
        if low <= numeric <= high:
            return chapter

    return "Unknown"

In [10]:
# Reimport src.features in case we just edited it (Jupyter caches modules)
import importlib
import src.features
importlib.reload(src.features)
from src.features import group_icd9

# Quick sanity check on the function before applying it
test_codes = ["250.83", "428.0", "486", "V58", "E885", "?", None, "nan"]
for code in test_codes:
    print(f"{repr(code):>12} → {group_icd9(code)}")

    '250.83' → Endocrine
     '428.0' → Circulatory
       '486' → Respiratory
       'V58' → V_code
      'E885' → E_code
         '?' → Unknown
        None → Missing
       'nan' → Unknown


In [11]:
# Apply chapter grouping to the three diagnosis columns
for col in ["diag_1", "diag_2", "diag_3"]:
    df[f"{col}_chapter"] = df[col].apply(group_icd9)

# Inspect the result for the primary diagnosis
print("diag_1_chapter distribution (primary admission diagnosis):")
print(df["diag_1_chapter"].value_counts())

diag_1_chapter distribution (primary admission diagnosis):
diag_1_chapter
Circulatory        21322
Endocrine           7599
Respiratory         6451
Digestive           6326
Symptoms            5503
Injury              4696
Musculoskeletal     4064
Genitourinary       3415
Neoplasms           2538
Skin                1780
Infectious          1685
Mental              1545
V_code               918
Nervous              858
Blood                652
Pregnancy            586
Congenital            41
Missing               10
E_code                 1
Name: count, dtype: int64


In [12]:
# Drop the raw ICD-9 columns — we'll use the chapter-grouped versions
df = df.drop(columns=["diag_1", "diag_2", "diag_3"])
print(f"Shape after dropping raw diagnosis codes: {df.shape}")

Shape after dropping raw diagnosis codes: (69990, 49)


### ICD-9 grouping into clinical chapters

`diag_1`, `diag_2`, and `diag_3` originally contained ~700 unique ICD-9 codes each. Using them as raw categorical features would explode dimensionality and produce categories with sample sizes too small to learn from.

We grouped each code into one of the **17 clinical chapters** of the ICD-9-CM hierarchy (Circulatory, Respiratory, Endocrine, etc.), plus two special categories:

- **V_code** — factors influencing health status (e.g., aftercare, V58)
- **E_code** — external causes of injury (e.g., fall, E885)

This reduces ~700 categories to ~19 per diagnosis slot, with clinically interpretable features (`diag_1_chapter`, etc.).

**Key clinical finding from `diag_1_chapter`:**

| Chapter | n | % |
|---|---|---|
| Circulatory | 21,322 | 30.5% |
| Endocrine | 7,599 | 10.9% |
| Respiratory | 6,451 | 9.2% |
| Digestive | 6,326 | 9.0% |

**Cardiovascular diagnoses outnumber endocrine ones by nearly 3:1** as primary reason for admission. This confirms one of the central truths of diabetology — diabetic patients are hospitalized far more often for *consequences* of diabetes (heart failure, MI, stroke, renal failure) than for metabolic decompensation of the disease itself (DKA, HHS, hypoglycemia). We expect `diag_1_chapter = Circulatory` to be a top predictor in the SHAP analysis (Notebook 04).

Note also that the `Symptoms` chapter (7.9%) captures ICD-9 codes 780–799 — admissions with ill-defined presentations (e.g., undifferentiated chest pain, syncope, fever of unknown origin) — a real and substantial clinical category, not a data quality artifact.

In [13]:
print(sorted(df.columns))

['A1Cresult', 'a1c_measured', 'acarbose', 'acetohexamide', 'admission_source_id', 'admission_type', 'age', 'change', 'chlorpropamide', 'diabetesMed', 'diag_1_chapter', 'diag_2_chapter', 'diag_3_chapter', 'discharge_disposition_id', 'gender', 'glimepiride', 'glimepiride-pioglitazone', 'glipizide', 'glipizide-metformin', 'glu_measured', 'glyburide', 'glyburide-metformin', 'insulin', 'max_glu_serum', 'medical_specialty', 'metformin', 'metformin-pioglitazone', 'metformin-rosiglitazone', 'miglitol', 'n_meds_changed', 'nateglinide', 'num_lab_procedures', 'num_medications', 'num_procedures', 'number_diagnoses', 'number_emergency', 'number_inpatient', 'number_outpatient', 'payer_code', 'pioglitazone', 'race', 'readmitted', 'repaglinide', 'rosiglitazone', 'time_in_hospital', 'tolazamide', 'tolbutamide', 'total_visits_prior', 'troglitazone']


In [14]:
# Create the binary target using the helper from src/features.py
from src.features import binarize_readmission

df["readm_30"] = binarize_readmission(df["readmitted"])

# Drop the original 3-class target — we're not using it
df = df.drop(columns=["readmitted"])

# Verify class balance
print("Target distribution (readm_30):")
print(df["readm_30"].value_counts())
print(f"\nPositive class rate: {df['readm_30'].mean()*100:.2f}%")

Target distribution (readm_30):
readm_30
0    63705
1     6285
Name: count, dtype: int64

Positive class rate: 8.98%


In [15]:
age_order = [
    "[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
    "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)",
]
age_to_int = {label: i for i, label in enumerate(age_order)}

df["age_ordinal"] = df["age"].map(age_to_int)
df = df.drop(columns=["age"])

print(df["age_ordinal"].value_counts().sort_index())
print(f"\nMissing in age_ordinal: {df['age_ordinal'].isna().sum()}")

age_ordinal
0      153
1      534
2     1121
3     2692
4     6828
5    12351
6    15689
7    17751
8    11110
9     1761
Name: count, dtype: int64

Missing in age_ordinal: 0


In [16]:
# Drop ID columns that served their purpose during cohort filtering
# but are too granular and overlapping to be useful as raw features.
df = df.drop(columns=["admission_source_id", "discharge_disposition_id"])
print(f"Shape: {df.shape}")

Shape: (69990, 47)


In [17]:
# Identify remaining categorical columns
categorical_cols = df.select_dtypes(include="object").columns.tolist()
print(f"One-hot encoding {len(categorical_cols)} categorical columns:")
print(categorical_cols)

# One-hot encode. drop_first=False keeps all dummies (better for tree models
# and for SHAP interpretability in Notebook 04).
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=False)

print(f"\nShape after encoding: {df_encoded.shape}")

One-hot encoding 33 categorical columns:
['race', 'gender', 'payer_code', 'medical_specialty', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'admission_type', 'diag_1_chapter', 'diag_2_chapter', 'diag_3_chapter']

Shape after encoding: (69990, 253)


/var/folders/3d/b6tlh05n7mj440lzzc7_nh0w0000gp/T/ipykernel_96823/3642669319.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include="object").columns.tolist()


In [18]:
# Final sanity checks before saving
print("=" * 60)
print("FINAL SANITY CHECKS")
print("=" * 60)

print(f"\nShape: {df_encoded.shape[0]:,} rows × {df_encoded.shape[1]} columns")

print(f"\nResidual missing values: {df_encoded.isna().sum().sum()}")
if df_encoded.isna().sum().sum() > 0:
    print("Columns with missing:")
    print(df_encoded.isna().sum()[df_encoded.isna().sum() > 0])

print(f"\nTarget positive rate: {df_encoded['readm_30'].mean()*100:.2f}%")

print(f"\nDtype summary:")
print(df_encoded.dtypes.value_counts())

FINAL SANITY CHECKS

Shape: 69,990 rows × 253 columns

Residual missing values: 0

Target positive rate: 8.98%

Dtype summary:
bool     239
int64     14
Name: count, dtype: int64


In [19]:
# Separate features from target, then save to data/processed/
from pathlib import Path

PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

X = df_encoded.drop(columns=["readm_30"])
y = df_encoded["readm_30"]

X.to_parquet(PROCESSED / "X.parquet", index=False)
y.to_frame().to_parquet(PROCESSED / "y.parquet", index=False)

print(f"Saved X: {X.shape}")
print(f"Saved y: shape ({len(y):,},), positive rate {y.mean()*100:.2f}%")
print(f"\nFiles in {PROCESSED}:")
for f in PROCESSED.iterdir():
    if not f.name.startswith("."):
        size_mb = f.stat().st_size / 1024**2
        print(f"  {f.name}: {size_mb:.2f} MB")

Saved X: (69990, 252)
Saved y: shape (69,990,), positive rate 8.98%

Files in ../data/processed:
  y.parquet: 0.01 MB
  X.parquet: 1.20 MB


---

## Summary — outputs and next step

### Pipeline executed

1. **Cohort definition**: excluded 2,423 expired/hospice encounters; deduplicated to first encounter per patient → **69,990 unique patients** (positive class rate: 8.98%)
2. **Dropped**: `weight` (97% missing), `examide` and `citoglipton` (zero variance), `encounter_id`, `patient_nbr`, `diag_1/2/3` (replaced by chapter groupings), `admission_source_id`, `discharge_disposition_id`
3. **Missing as category**: `A1Cresult`, `max_glu_serum`, `medical_specialty`, `payer_code`, `race`; consolidated disguised missingness in `admission_type`
4. **Engineered features**:
    - `a1c_measured`, `glu_measured` (binary measurement flags)
    - `total_visits_prior` (composite utilization)
    - `n_meds_changed` (active diabetes management signal)
    - `diag_1_chapter`, `diag_2_chapter`, `diag_3_chapter` (ICD-9 grouped into 19 clinical chapters)
5. **Encoded**: `age` as ordered ordinal (0–9), all categoricals as one-hot, target binarized via `binarize_readmission`
6. **Saved** modeling-ready dataset to `data/processed/`

### Output files

| File | Shape | Size |
|---|---|---|
| `X.parquet` | 69,990 × 252 | 1.20 MB |
| `y.parquet` | 69,990 × 1 | 0.01 MB |

### Note on class imbalance

The positive class rate is **8.98%** — slightly lower than the 11.16% observed in the raw dataset. The drop reflects deduplication: patients with multiple encounters tend to have higher readmission rates than single-encounter patients, so keeping only the first encounter per patient reduces the positive rate. This is the correct cohort for modeling — it represents the population at the moment of the **index admission**, before any subsequent observation.

### Next: Notebook 03 — Modeling

- Train/test split (stratified by `readm_30`)
- Three models: Logistic Regression (interpretable baseline), Random Forest, XGBoost
- Address imbalance via `class_weight='balanced'` and/or SMOTE
- Evaluate with AUC-ROC, precision-recall AUC, and recall on the positive class